### 🤖 Feature Engineering para Modelo de Risco de Defasagem

**Base:** `df_base_2.parquet`

**Objetivo:** Construir o dataset de ML para prever se um aluno estará defasado (IAN < 10) no ano seguinte.

**Decisões baseadas na Análise Exploratória de dados:**

**Target:**
- `defasado_futuro`: IAN < 10 no ano seguinte (binário)

**Features selecionadas:**
- Indicadores Diretos: IAN
- Indicadores Normalizados: `IDA_norm`, `IEG_norm`, `IPS_norm`, `IPP_norm`, `IPV_norm`

- Contextuais: `nivel`, `tipo_escola`, `primeiro_ano`
  - `nivel` (nova coluna `nivel_num`) e `tipo_escola` foram mapeados para númericos
- Removidos por não constribuirem: `gap_iaa_ida`, `pedra_num`, `ips_no_piso`, `IAA_norm`


**Tratamentos:**
- Normalização por percentil dentro do ano (IPS, IAA, IPP têm escalas diferentes entre anos)
- Não usar INDE (data leakage — calculado a partir dos indicadores)

**Validação:**
- Temporal: treinar com pares 2022->2023, testar com pares 2023->2024

In [33]:
import pandas as pd
import numpy as np
import warnings
import json

warnings.filterwarnings('ignore')

df = pd.read_parquet('../../../data/db/01_silver_processed/df_base_2.parquet')

indicadores = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV']

VERSAO = 'v3'

print(f'df_base: {df.shape[0]} linhas | {df.shape[1]} colunas')

df_base: 3030 linhas | 31 colunas


### Construção dos pares ano-a-ano

In [34]:
# Construir pares ano-a-ano: features do ano atual + target do ano seguinte
pares = []

for ra, grupo in df.groupby('RA'):
    grupo = grupo.sort_values('ano')
    for i in range(len(grupo) - 1):
        atual = grupo.iloc[i]
        proximo = grupo.iloc[i + 1]
        
        # Target: defasado no ano seguinte
        if pd.isna(proximo['IAN']):
            continue
            
        par = {
            'RA': ra,
            'ano_atual': int(atual['ano']),
            'ano_proximo': int(proximo['ano']),
            'target': int(proximo['IAN'] < 10),
        }
        
        # Features — indicadores diretos
        for ind in indicadores:
            par[ind] = atual[ind]
        
        # Features — contextuais
        par['nivel'] = atual['nivel']
        par['pedra'] = atual['pedra']
        par['instituicao'] = atual['instituicao']
        par['ano_ingresso'] = atual['ano_ingresso']
        
        pares.append(par)

df_pares = pd.DataFrame(pares)

print(f'=== Pares construídos ===')
print(f'|_  Total: {len(df_pares)}')
print(f'|_  2022->2023: {(df_pares["ano_atual"] == 2022).sum()}')
print(f'|_  2023->2024: {(df_pares["ano_atual"] == 2023).sum()}')
print(f'\n=== Distribuição do target ===')
print(f'|_  Defasado (1): {df_pares["target"].sum()} ({df_pares["target"].mean()*100:.1f}%)')
print(f'|_  Adequado (0): {(df_pares["target"] == 0).sum()} ({(df_pares["target"] == 0).mean()*100:.1f}%)')

# Por transição
print(f'\n=== Target por transição ===')
for ano in [2022, 2023]:
    sub = df_pares[df_pares['ano_atual'] == ano]
    print(f'|_  {ano}->{ano+1}: {sub["target"].mean()*100:.1f}% defasados (n={len(sub)})')

=== Pares construídos ===
|_  Total: 1369
|_  2022->2023: 604
|_  2023->2024: 765

=== Distribuição do target ===
|_  Defasado (1): 675 (49.3%)
|_  Adequado (0): 694 (50.7%)

=== Target por transição ===
|_  2022->2023: 60.8% defasados (n=604)
|_  2023->2024: 40.3% defasados (n=765)


### Features derivadas e tratamentos

In [35]:
df_pares['instituicao'].unique()

array(['Escola Pública', 'Privada *Parcerias com Bolsa 100%', 'Pública',
       'Rede Decisão', 'Privada - Programa de Apadrinhamento', 'Privada',
       'Concluiu o 3º EM', 'Privada - Pagamento por *Empresa Parceira',
       'Nenhuma das opções acima', 'Escola JP II',
       'Privada - Programa de apadrinhamento'], dtype=object)

In [36]:
# Features derivadas
df_pares['gap_iaa_ida'] = df_pares['IAA'] - df_pares['IDA']
df_pares['resiliencia'] = df_pares['IPP'] - df_pares['IAN']

# Feature: primeiro ano no programa
df_pares['primeiro_ano'] = (df_pares['ano_ingresso'] == df_pares['ano_atual']).astype(int)

# Feature: tipo de escola (pública vs privada)
def classificar_escola(x):
    x_upper = str(x).upper()
    if 'PÚBLICA' in x_upper or 'PUBLICA' in x_upper:
        return 0
    elif 'PRIVADA' in x_upper or 'BOLSISTA' in x_upper or 'DECISÃO' in x_upper or 'JP II' in x_upper:
        return 1
    else:
        return np.nan

df_pares['tipo_escola'] = df_pares['instituicao'].apply(classificar_escola)

# Feature: flag IPS no piso (valor mínimo do ano)
# df_pares['ips_no_piso'] = 0
# df_pares.loc[(df_pares['ano_atual'] == 2022) & (df_pares['IPS'] == 2.50), 'ips_no_piso'] = 1
# df_pares.loc[(df_pares['ano_atual'] == 2023) & (df_pares['IPS'] == 2.52), 'ips_no_piso'] = 1

df_pares['ips_no_piso'] = df_pares.groupby('ano_atual')['IPS'].transform(
    lambda x: (x == x.min()).astype(int)
)

# Normalização por percentil dentro do ano (remove efeito de escala entre anos)
indicadores_normalizar = ['IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV']

for ind in indicadores_normalizar:
    col_norm = f'{ind}_norm'
    df_pares[col_norm] = df_pares.groupby('ano_atual')[ind].transform(
        lambda x: x.rank(pct=True)
    )

# Converter nivel para numérico (ALFA = 0)
df_pares['nivel_num'] = df_pares['nivel'].replace('ALFA', '0').astype(float)

# Converter pedra para numérico (ordem natural: Quartzo < Ágata < Ametista < Topázio)
df_pares['pedra_num'] = df_pares['pedra'].map({
    'Quartzo': 0, 'Ágata': 1, 'Ametista': 2, 'Topázio': 3
})

print('=== Features criadas ===')
print(f'|_ Derivadas: gap_iaa_ida, resiliencia, primeiro_ano, tipo_escola, ips_no_piso')
print(f'|_ Normalizadas: {[f"{ind}_norm" for ind in indicadores_normalizar]}')
print(f'\n=== Verificação ===')
print(f'|_ Nulos por feature:')
features_todas = indicadores + ['gap_iaa_ida', 'resiliencia', 'primeiro_ano', 'tipo_escola', 
                                 'ips_no_piso'] + [f'{ind}_norm' for ind in indicadores_normalizar]
for feat in features_todas:
    nulos = df_pares[feat].isna().sum()
    if nulos > 0:
        print(f'|  |_ {feat}: {nulos} ({nulos/len(df_pares)*100:.1f}%)')
print(f'|')
print(f'|_ Shape: {df_pares.shape}')
print(f'|_ Colunas: {list(df_pares.columns)}')

=== Features criadas ===
|_ Derivadas: gap_iaa_ida, resiliencia, primeiro_ano, tipo_escola, ips_no_piso
|_ Normalizadas: ['IDA_norm', 'IEG_norm', 'IAA_norm', 'IPS_norm', 'IPP_norm', 'IPV_norm']

=== Verificação ===
|_ Nulos por feature:
|  |_ IDA: 72 (5.3%)
|  |_ IEG: 72 (5.3%)
|  |_ IAA: 59 (4.3%)
|  |_ IPS: 62 (4.5%)
|  |_ IPP: 72 (5.3%)
|  |_ IPV: 72 (5.3%)
|  |_ gap_iaa_ida: 72 (5.3%)
|  |_ resiliencia: 72 (5.3%)
|  |_ tipo_escola: 6 (0.4%)
|  |_ IDA_norm: 72 (5.3%)
|  |_ IEG_norm: 72 (5.3%)
|  |_ IAA_norm: 59 (4.3%)
|  |_ IPS_norm: 62 (4.5%)
|  |_ IPP_norm: 72 (5.3%)
|  |_ IPV_norm: 72 (5.3%)
|
|_ Shape: (1369, 28)
|_ Colunas: ['RA', 'ano_atual', 'ano_proximo', 'target', 'IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV', 'nivel', 'pedra', 'instituicao', 'ano_ingresso', 'gap_iaa_ida', 'resiliencia', 'primeiro_ano', 'tipo_escola', 'ips_no_piso', 'IDA_norm', 'IEG_norm', 'IAA_norm', 'IPS_norm', 'IPP_norm', 'IPV_norm', 'nivel_num', 'pedra_num']


### Preparação do dataset final para ML

In [37]:
# Definir features finais para o modelo
# Usar versões normalizadas dos indicadores + features derivadas + contextuais
# Features removidas da v2: 'IAA_norm', 'gap_iaa_ida', 'pedra_num', 'ips_no_piso'
features_modelo = [
    # Indicadores normalizados por ano ( removida)
    'IDA_norm', 'IEG_norm', 'IPS_norm', 'IPP_norm', 'IPV_norm',
    # IAN direto (é discreto: 2.5, 5.0, 10.0 — não precisa normalizar)
    'IAN',
    # Features derivadas
    'resiliencia', 
    # Contextuais
    'primeiro_ano', 
    'nivel_num', 'tipo_escola', # adicionadas como numéricas para evitar encoding e testar se ajudam o modelo
]

# Categoricas (precisam de encoding)
# features_cat = ['nivel', 'tipo_escola', 'pedra']
features_cat = []

# Separar treino (2022→2023) e teste (2023→2024)
df_treino = df_pares[df_pares['ano_atual'] == 2022].copy()
df_teste = df_pares[df_pares['ano_atual'] == 2023].copy()

print(f'=== Dataset final ===')
print(f'|_  Treino (2022->2023): {len(df_treino)} pares, target={df_treino["target"].mean()*100:.1f}% defasados')
print(f'|_  Teste (2023->2024): {len(df_teste)} pares, target={df_teste["target"].mean()*100:.1f}% defasados')

print(f'\n=== Features numéricas ({len(features_modelo)}): ===')
for f in features_modelo:
    print(f'|_  {f}')

print(f'\n=== Features categóricas ({len(features_cat)}): ===')
for f in features_cat:
    valores = df_pares[f].unique()
    print(f'|_  {f}: {len(valores)} valores — {sorted([str(v) for v in valores if pd.notna(v)])}')

=== Dataset final ===
|_  Treino (2022->2023): 604 pares, target=60.8% defasados
|_  Teste (2023->2024): 765 pares, target=40.3% defasados

=== Features numéricas (10): ===
|_  IDA_norm
|_  IEG_norm
|_  IPS_norm
|_  IPP_norm
|_  IPV_norm
|_  IAN
|_  resiliencia
|_  primeiro_ano
|_  nivel_num
|_  tipo_escola

=== Features categóricas (0): ===


In [38]:
df_pares.head()

,RA,ano_atual,ano_proximo,target,IAN,IDA,IEG,IAA,IPS,IPP,...,tipo_escola,ips_no_piso,IDA_norm,IEG_norm,IAA_norm,IPS_norm,IPP_norm,IPV_norm,nivel_num,pedra_num
0,RA-1,2022,2023,0,5.0,4.0,4.1,8.3,5.60,7.96875,...,0.0,0,0.126656,0.020695,0.316225,0.189570,0.860927,0.385762,7.0,0.0
1,RA-1,2023,2024,0,10.0,NaN,NaN,NaN,NaN,NaN,...,1.0,0,NaN,NaN,NaN,NaN,NaN,NaN,8.0,NaN
2,RA-1000,2023,2024,0,10.0,7.0,9.4,8.5,3.77,6.25000,...,0.0,0,0.529582,0.627706,0.486544,0.359886,0.112554,0.821068,0.0,2.0
3,RA-1001,2023,2024,1,5.0,7.8,9.1,9.0,7.52,7.50000,...,0.0,0,0.714286,0.487734,0.642351,0.830014,0.388167,0.877345,0.0,3.0
4,RA-1002,2023,2024,1,5.0,7.0,9.7,9.0,7.52,6.25000,...,0.0,0,0.529582,0.810245,0.642351,0.830014,0.112554,0.821068,0.0,2.0


In [39]:
df_pares['primeiro_ano'].value_counts()

primeiro_ano
0    847
1    522
Name: count, dtype: int64

In [40]:
df_pares['features'] = json.dumps(features_modelo)
df_pares['features']

0       ["IDA_norm", "IEG_norm", "IPS_norm", "IPP_norm...
1       ["IDA_norm", "IEG_norm", "IPS_norm", "IPP_norm...
2       ["IDA_norm", "IEG_norm", "IPS_norm", "IPP_norm...
3       ["IDA_norm", "IEG_norm", "IPS_norm", "IPP_norm...
4       ["IDA_norm", "IEG_norm", "IPS_norm", "IPP_norm...
                              ...                        
1364    ["IDA_norm", "IEG_norm", "IPS_norm", "IPP_norm...
1365    ["IDA_norm", "IEG_norm", "IPS_norm", "IPP_norm...
1366    ["IDA_norm", "IEG_norm", "IPS_norm", "IPP_norm...
1367    ["IDA_norm", "IEG_norm", "IPS_norm", "IPP_norm...
1368    ["IDA_norm", "IEG_norm", "IPS_norm", "IPP_norm...
Name: features, Length: 1369, dtype: object

In [41]:
# Salvar para uso no notebook 13
df_pares.to_parquet(f'../../../data/db/01_silver_processed/df_ml_pares_{VERSAO}.parquet', index=False)
print(f'\n✅ Dataset salvo em data/db/01_silver_processed/df_ml_pares_{VERSAO}.parquet')


✅ Dataset salvo em data/db/01_silver_processed/df_ml_pares_v3.parquet
